In [0]:
"""
id: python_4
template: python
templateVersion: 1.0.0
name: API_ customers
position:
  x: 985.2830901016546
  y: 406.776495992537
description:
  text: Load customer data from an online CSV file or use provided data.
  hash: c678b4d4
previewCodeHash: e8bea9d68523965b
previewMode: "1000"
config:
  code: |
    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import requests
        import pandas as pd
        from io import StringIO
        url="https://raw.githubusercontent.com/Mannie9/Databricks_Lakeflow/refs/heads/main/Dataset/customers.csv"

        response = requests.get(url)
        csv_data = response.content.decode('utf-8')

        df = pd.read_csv(StringIO(csv_data)) 
        result = spark.createDataFrame(df)
input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import requests
        import pandas as pd
        from io import StringIO
        url="https://raw.githubusercontent.com/Mannie9/Databricks_Lakeflow/refs/heads/main/Dataset/customers.csv"

        response = requests.get(url)
        csv_data = response.content.decode('utf-8')

        df = pd.read_csv(StringIO(csv_data)) 
        result = spark.createDataFrame(df)

    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_4.result"] = out["result"]

In [0]:
"""
id: python_5
template: python
templateVersion: 1.0.0
name: API_shipments
position:
  x: 1300.6902587404027
  y: 475.54278719870416
description:
  text: Load data from a URL or use provided data.
  hash: 7f0b9d51
previewCodeHash: 6d423d52472fe9af
previewMode: "1000"
config:
  code: |+
    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import requests
        import pandas as pd
        from io import StringIO
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/shipments.csv"

        response = requests.get(url)
        csv_data = response.content.decode('utf-8')

        df = pd.read_csv(StringIO(csv_data)) 
        result = spark.createDataFrame(df)

input: []
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    if inputs.get("data"):
        result = inputs["data"][0]
    else:
        import requests
        import pandas as pd
        from io import StringIO
        url="https://raw.githubusercontent.com/anshlambagit/Databricks_Lakeflow_Designer/refs/heads/main/shipments.csv"

        response = requests.get(url)
        csv_data = response.content.decode('utf-8')

        df = pd.read_csv(StringIO(csv_data)) 
        result = spark.createDataFrame(df)

    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {}
out = run(config, inputs, spark)
ctx["python_5.result"] = out["result"]

In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: orders
position:
  x: 693.6605173330062
  y: 182.63810210005227
description:
  text: Read all data from the orders table.
  hash: fa64d78c
previewCodeHash: 2fc0e8560d434db4
previewMode: "1000"
config:
  table_source:
    tableName: lakeflow_designer.raw.orders
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "lakeflow_designer.raw.orders"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]

In [0]:
"""
id: source_1
template: source
templateVersion: 2.0.0
name: order_items
position:
  x: 692.1375862854338
  y: 339.8215504806308
description:
  text: Get all data from order_items table.
  hash: 94d2fad9
previewCodeHash: fabbfcb3eb484e3c
previewMode: "1000"
config:
  table_source:
    tableName: lakeflow_designer.raw.order_items
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "lakeflow_designer.raw.order_items"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_1.data"] = out["data"]

In [0]:
"""
id: source_18
template: source
templateVersion: 2.0.0
name: reviews
position:
  x: 1580.9549588315026
  y: 606.8484253756384
description:
  text: Load all data from the reviews table.
  hash: 94396eb0
previewCodeHash: 382bae1773b99d22
previewMode: "1000"
config:
  table_source:
    tableName: lakeflow_designer.raw.reviews
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "lakeflow_designer.raw.reviews"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_18.data"] = out["data"]

In [0]:
"""
id: source_2
template: source
templateVersion: 2.0.0
name: products
position:
  x: 1578.362218226547
  y: 225.5677305853598
description:
  text: Retrieve all data from the products table.
  hash: cbbf03b7
previewCodeHash: 67844fa8de90e23f
previewMode: "1000"
config:
  table_source:
    tableName: lakeflow_designer.raw.products
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "lakeflow_designer.raw.products"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_2.data"] = out["data"]

In [0]:
"""
id: join_3
template: join
templateVersion: 1.0.0
name: OrderJoinOrderItems
position:
  x: 995.7248274291323
  y: 259.66564514786995
description:
  text: Left join on order_id column, keeping specific columns from both sides.
  hash: dfe9435d
previewCodeHash: 2f6bc71bc0725f50
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.order_id = right.order_id
  expressions:
    - "`left`.order_id"
    - "`left`.customer_id"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`right`.order_item_id"
    - "`right`.product_id"
    - "`right`.quantity"
    - "`right`.unit_price"
    - "`right`.discount_pct"
    - "`right`.line_total"
input:
  - node: source_0
    input_port: left
    output_port: data
  - node: source_1
    input_port: right
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.order_id = right.order_id",
    "expressions": [
        "`left`.order_id",
        "`left`.customer_id",
        "`left`.order_date",
        "`left`.order_status",
        "`right`.order_item_id",
        "`right`.product_id",
        "`right`.quantity",
        "`right`.unit_price",
        "`right`.discount_pct",
        "`right`.line_total"
    ]
}
inputs = {
    "left": ctx["source_0.data"],
    "right": ctx["source_1.data"]
}
out = run(config, inputs, spark)
ctx["join_3.joined_data"] = out["joined_data"]

In [0]:
"""
id: ai_function_19
template: ai_function
templateVersion: 3.0.0
name: Sentimental Analysis
position:
  x: 1840.9549588315026
  y: 606.8484253756384
description:
  text: Add sentiment analysis results while keeping all original columns.
  hash: e47555ba
previewCodeHash: d5755c0b081866b1
previewMode: "1000"
config:
  expressions:
    - ai_analyze_sentiment(review_body) `Sentiment`
  keep_all_columns: true
input:
  - node: source_18
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any, List
import hashlib

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]

    # Table-valued AI functions (ai_forecast, ...) live in the FROM
    # clause and have no SELECT-list shape, so we run them via
    # spark.sql and bind the upstream DataFrame as a session-scoped
    # temp view substituted in for the
    # __lakebuilder_ai_function_input__ placeholder identifier
    # (which is a real SQL identifier so the persisted statement
    # round-trips through the SQL parser when the cell reloads).
    #
    # spark.sql returns a lazy DataFrame: the view name is resolved
    # by Spark at action time (preview limit/collect), not when
    # spark.sql is called. We therefore deliberately do NOT drop
    # the temp view here — dropping it would leave the lazy plan
    # pointing at a missing relation and trigger TABLE_OR_VIEW_NOT
    # _FOUND when the downstream action fires.
    #
    # The view name is derived deterministically from (tvf_sql,
    # id(df)) so reruns of the same cell reuse the same name and
    # createOrReplaceTempView keeps the session catalog bounded at
    # one entry per (cell × upstream) instead of growing one entry
    # per run. id(df) is included to keep two cells that happen to
    # have identical tvf_sql but distinct upstream DataFrames from
    # clobbering each other's bindings.
    tvf_sql: str = config.get("tvf_sql") or ""
    if tvf_sql:
        key = f"{tvf_sql}\x00{id(df)}".encode("utf-8")
        view_name = f"lakebuilder_ai_fn_{hashlib.sha256(key).hexdigest()[:12]}"
        df.createOrReplaceTempView(view_name)
        sql = tvf_sql.replace("__lakebuilder_ai_function_input__", view_name)
        return {"ai_data": spark.sql(sql)}

    expressions: List[str] = config.get("expressions", [])
    if not expressions:
        return {"ai_data": df}

    keep_all_columns: bool = config.get("keep_all_columns", True)
    if keep_all_columns:
        return {"ai_data": df.selectExpr(*expressions, "*")}
    return {"ai_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "ai_analyze_sentiment(review_body) `Sentiment`"
    ],
    "keep_all_columns": True
}
inputs = {
    "data": ctx["source_18.data"]
}
out = run(config, inputs, spark)
ctx["ai_function_19.ai_data"] = out["ai_data"]

In [0]:
"""
id: join_6
template: join
templateVersion: 1.0.0
name: OrderItemsJoinCustomer
position:
  x: 1299.9396466952978
  y: 334.7834279853261
description:
  text: Left join tables on customer_id and keep selected columns from both sides.
  hash: 64a43031
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.customer_id = right.customer_id
  expressions:
    - "`left`.order_id"
    - "`left`.customer_id"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`left`.order_item_id"
    - "`left`.product_id"
    - "`left`.quantity"
    - "`left`.unit_price"
    - "`left`.discount_pct"
    - "`left`.line_total"
    - "`right`.customer_name"
    - "`right`.email"
    - "`right`.phone"
    - "`right`.city"
    - "`right`.state"
    - "`right`.country"
    - "`right`.created_date"
input:
  - node: join_3
    input_port: left
    output_port: joined_data
  - node: python_4
    input_port: right
    output_port: result
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.customer_id = right.customer_id",
    "expressions": [
        "`left`.order_id",
        "`left`.customer_id",
        "`left`.order_date",
        "`left`.order_status",
        "`left`.order_item_id",
        "`left`.product_id",
        "`left`.quantity",
        "`left`.unit_price",
        "`left`.discount_pct",
        "`left`.line_total",
        "`right`.customer_name",
        "`right`.email",
        "`right`.phone",
        "`right`.city",
        "`right`.state",
        "`right`.country",
        "`right`.created_date"
    ]
}
inputs = {
    "left": ctx["join_3.joined_data"],
    "right": ctx["python_4.result"]
}
out = run(config, inputs, spark)
ctx["join_6.joined_data"] = out["joined_data"]

In [0]:
"""
id: output_20
template: output
templateVersion: 1.0.0
name: lakeflow_designer.enr.SentimentalAnalysis
position:
  x: 2100.9549588315026
  y: 606.8484253756384
description:
  text: Save data to the specified table, replacing existing data.
  hash: 2ecf7d86
previewMode: "1000"
config:
  catalog: lakeflow_designer
  schema: enr
  table_name: SentimentalAnalysis
input:
  - node: ai_function_19
    input_port: data
    output_port: ai_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "lakeflow_designer",
    "schema": "enr",
    "table_name": "SentimentalAnalysis"
}
inputs = {
    "data": ctx["ai_function_19.ai_data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: join_7
template: join
templateVersion: 1.0.0
name: Order+CustomerJoinsShipments
position:
  x: 1575.885469738215
  y: 398.70660768362615
description:
  text: Left join on order_id to combine order details with shipment information and keep specific columns from both tables.
  hash: 30fcc890
previewCodeHash: 97e12c5aa1ae99c1
previewMode: "1000"
config:
  join_type: left
  join_conditions: left.order_id = right.order_id
  expressions:
    - "`left`.order_id"
    - "`left`.customer_id"
    - "`left`.order_date"
    - "`left`.order_status"
    - "`left`.order_item_id"
    - "`left`.product_id"
    - "`left`.quantity"
    - "`left`.unit_price"
    - "`left`.discount_pct"
    - "`left`.line_total"
    - "`left`.customer_name"
    - "`left`.email"
    - "`left`.phone"
    - "`left`.city"
    - "`left`.state"
    - "`left`.country"
    - "`left`.created_date"
    - "`right`.shipment_id"
    - "`right`.ship_date"
    - "`right`.ship_mode"
    - "`right`.shipping_cost"
input:
  - node: join_6
    input_port: left
    output_port: joined_data
  - node: python_5
    input_port: right
    output_port: result
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.order_id = right.order_id",
    "expressions": [
        "`left`.order_id",
        "`left`.customer_id",
        "`left`.order_date",
        "`left`.order_status",
        "`left`.order_item_id",
        "`left`.product_id",
        "`left`.quantity",
        "`left`.unit_price",
        "`left`.discount_pct",
        "`left`.line_total",
        "`left`.customer_name",
        "`left`.email",
        "`left`.phone",
        "`left`.city",
        "`left`.state",
        "`left`.country",
        "`left`.created_date",
        "`right`.shipment_id",
        "`right`.ship_date",
        "`right`.ship_mode",
        "`right`.shipping_cost"
    ]
}
inputs = {
    "left": ctx["join_6.joined_data"],
    "right": ctx["python_5.result"]
}
out = run(config, inputs, spark)
ctx["join_7.joined_data"] = out["joined_data"]

In [0]:
"""
id: join_8
template: join
templateVersion: 1.0.0
name: OBT
position:
  x: 1858.1603218449873
  y: 309.32893820183676
description:
  text: Right join on product_id, keeping specific columns from both sides.
  hash: 13df1e38
previewCodeHash: ad3eb36b23515aa7
previewMode: "1000"
config:
  join_type: right
  join_conditions: left.product_id = right.product_id
  expressions:
    - "`right`.order_id"
    - "`right`.customer_id"
    - "`right`.order_date"
    - "`right`.order_status"
    - "`right`.order_item_id"
    - "`right`.product_id"
    - "`right`.quantity"
    - "`right`.unit_price"
    - "`right`.discount_pct"
    - "`right`.line_total"
    - "`right`.customer_name"
    - "`right`.email"
    - "`right`.phone"
    - "`right`.city"
    - "`right`.state"
    - "`right`.country"
    - "`right`.created_date"
    - "`right`.shipment_id"
    - "`right`.ship_date"
    - "`right`.ship_mode"
    - "`right`.shipping_cost"
    - "`left`.product_name"
    - "`left`.category"
    - "`left`.subcategory"
input:
  - node: source_2
    input_port: left
    output_port: data
  - node: join_7
    input_port: right
    output_port: joined_data
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "right",
    "join_conditions": "left.product_id = right.product_id",
    "expressions": [
        "`right`.order_id",
        "`right`.customer_id",
        "`right`.order_date",
        "`right`.order_status",
        "`right`.order_item_id",
        "`right`.product_id",
        "`right`.quantity",
        "`right`.unit_price",
        "`right`.discount_pct",
        "`right`.line_total",
        "`right`.customer_name",
        "`right`.email",
        "`right`.phone",
        "`right`.city",
        "`right`.state",
        "`right`.country",
        "`right`.created_date",
        "`right`.shipment_id",
        "`right`.ship_date",
        "`right`.ship_mode",
        "`right`.shipping_cost",
        "`left`.product_name",
        "`left`.category",
        "`left`.subcategory"
    ]
}
inputs = {
    "left": ctx["source_2.data"],
    "right": ctx["join_7.joined_data"]
}
out = run(config, inputs, spark)
ctx["join_8.joined_data"] = out["joined_data"]

In [0]:
"""
id: aggregate_9
template: aggregate
templateVersion: 2.0.0
name: OrdersbyCity
position:
  x: 2151.0765321186423
  y: 435.121527144449
description:
  text: Group data by city and count orders, also calculate total amount spent.
  hash: 2796d5b6
previewCodeHash: 7dc0221f74bbbf87
previewMode: "1000"
config:
  group_bys:
    - expr: city
      type: column
  aggregations:
    - columnExpr:
        expr: order_id
        type: column
      fn: COUNT
      alias: TotalOrders
    - columnExpr:
        expr: ROUND(SUM(unit_price),2)
        type: expr
      fn: "-"
      alias: TotalAmount
input:
  - node: join_8
    input_port: data
    output_port: joined_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "city",
            "type": "column"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "order_id",
                "type": "column"
            },
            "fn": "COUNT",
            "alias": "TotalOrders",
            "withAsKeyword": None
        },
        {
            "columnExpr": {
                "expr": "ROUND(SUM(unit_price),2)",
                "type": "expr"
            },
            "fn": "-",
            "alias": "TotalAmount",
            "withAsKeyword": None
        }
    ]
}
inputs = {
    "data": ctx["join_8.joined_data"]
}
out = run(config, inputs, spark)
ctx["aggregate_9.aggregated_data"] = out["aggregated_data"]

In [0]:
"""
id: prepare_order_date_parts
template: prepare
templateVersion: 1.0.0
name: PrepareOrderDateParts
position:
  x: 2126.1891728367696
  y: 178.96019536908494
previewCodeHash: cc8c876333853e29
previewMode: "1000"
config:
  actions:
    - type: formula
      target: order_year
      expression: YEAR(order_date)
    - type: formula
      target: order_month
      expression: MONTH(order_date)
    - type: formula
      target: month_year
      expression: CONCAT(CAST(MONTH(order_date) AS STRING), '-', CAST(YEAR(order_date) AS STRING))
input:
  - node: join_8
    input_port: data
    output_port: joined_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "order_year",
            "expression": "YEAR(order_date)"
        },
        {
            "type": "formula",
            "target": "order_month",
            "expression": "MONTH(order_date)"
        },
        {
            "type": "formula",
            "target": "month_year",
            "expression": "CONCAT(CAST(MONTH(order_date) AS STRING), '-', CAST(YEAR(order_date) AS STRING))"
        }
    ]
}
inputs = {
    "data": ctx["join_8.joined_data"]
}
out = run(config, inputs, spark)
ctx["prepare_order_date_parts.prepared_data"] = out["prepared_data"]

In [0]:
"""
id: sort_10
template: sort
templateVersion: 1.0.0
name: Sorted
position:
  x: 2411.0765321186423
  y: 435.121527144449
description:
  text: Sort data by TotalAmount from highest to lowest.
  hash: 6b4fb3d3
previewCodeHash: 6fde54dd4c90340f
previewMode: "1000"
config:
  sort_expressions:
    - columnExpr:
        expr: TotalAmount
        type: column
      sortBy: DESC
input:
  - node: aggregate_9
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "TotalAmount",
                "type": "column"
            },
            "sortBy": "DESC"
        }
    ]
}
inputs = {
    "data": ctx["aggregate_9.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["sort_10.sorted_data"] = out["sorted_data"]

In [0]:
"""
id: aggregate_monthly_order_status
template: aggregate
templateVersion: 2.0.0
name: MonthlyOrderStatusCounts
position:
  x: 2407.4551693702597
  y: 274.2219127149878
description:
  text: Group data by month of order date, order ID, and order status, then count orders for each group.
  hash: bfcce26a
previewCodeHash: 90a2e4b57b8854e5
previewMode: "1000"
config:
  group_bys:
    - expr: month_year
      type: column
    - expr: order_status
      type: column
  aggregations:
    - columnExpr:
        expr: order_id
        type: column
      fn: COUNT
      alias: OrderCount
input:
  - node: prepare_order_date_parts
    input_port: data
    output_port: prepared_data
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

DEFAULT_PERCENTILE = 0.5

DEFAULT_CONCAT_SEPARATOR = ", "

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
            "FIRST": F.first,
            "LAST": F.last,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = agg_fn(arg)
        elif fn == "-" or fn == "_":
            col = F.expr(raw_expr) if col_expr.get("type") == "expr" else F.col(raw_expr)
        elif fn == "PERCENTILE":
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        elif fn == "CONCAT":
            raw_sep = agg_def.get("separator")
            sep = raw_sep if isinstance(raw_sep, str) else DEFAULT_CONCAT_SEPARATOR
            concat_arg = F.expr(raw_expr) if col_expr.get("type") == "expr" else raw_expr
            col = F.concat_ws(sep, F.collect_list(concat_arg))
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "month_year",
            "type": "column"
        },
        {
            "expr": "order_status",
            "type": "column"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "order_id",
                "type": "column"
            },
            "fn": "COUNT",
            "alias": "OrderCount",
            "withAsKeyword": None
        }
    ]
}
inputs = {
    "data": ctx["prepare_order_date_parts.prepared_data"]
}
out = run(config, inputs, spark)
ctx["aggregate_monthly_order_status.aggregated_data"] = out["aggregated_data"]

In [0]:
"""
id: output_11
template: output
templateVersion: 1.0.0
name: lakeflow_designer.enr.AggregatedOrders
position:
  x: 2671.0765321186423
  y: 435.121527144449
description:
  text: Save data by replacing any existing table with this new table.
  hash: 42f70afd
previewMode: "1000"
config:
  catalog: lakeflow_designer
  schema: enr
  table_name: AggregatedOrders
input:
  - node: sort_10
    input_port: data
    output_port: sorted_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "lakeflow_designer",
    "schema": "enr",
    "table_name": "AggregatedOrders"
}
inputs = {
    "data": ctx["sort_10.sorted_data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: prepare_month_year_parts
template: prepare
templateVersion: 1.0.0
name: PrepareMonthYearParts
position:
  x: 2697.3541641704937
  y: 106.49218843606475
description:
  text: Split month_year into parts and extract numeric year and month for sorting.
  hash: 71641be5
previewCodeHash: 8f5f80caed822288
previewMode: "1000"
config:
  actions:
    - type: formula
      target: month_year_parts
      expression: SPLIT(month_year, '-')
    - type: formula
      target: sort_year
      expression: CAST(element_at(month_year_parts, 2) AS INT)
    - type: formula
      target: sort_month
      expression: CAST(element_at(month_year_parts, 1) AS INT)
input:
  - node: aggregate_monthly_order_status
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Any, Callable, Dict, List
import pyspark.sql.functions as F

TEXT_CASE_FUNCTIONS: Dict[str, Callable] = {
    "lower": F.lower,
    "upper": F.upper,
    "title": F.initcap,
}

TRIM_FUNCTIONS: Dict[str, Callable] = {
    "both": F.trim,
    "left": F.ltrim,
    "right": F.rtrim,
}

VALID_MATCH_MODES = {"exact", "contains", "prefix", "suffix", "regex"}

def _require_column(df, column: str, action_type: str) -> None:
    if not column:
        raise ValueError(f"{action_type}: 'column' is required")
    if column not in df.columns:
        raise ValueError(
            f"{action_type}: column '{column}' not found in input data. "
            f"Available columns: {df.columns}"
        )

def _column_dtype(df, column: str) -> str:
    return df.schema[column].dataType.simpleString()

def _quoted_ident(name: str) -> str:
    return "`" + name.replace("`", "``") + "`"

def _col(name: str):
    """Reference a column by exact name.

    F.col parses its argument as a column expression, so a name containing
    a '.' (e.g. "a.b") is otherwise read as struct-field access and fails
    even though the column exists. Backtick-quoting forces an exact-name
    lookup; quoting plain names is harmless.
    """
    return F.col(_quoted_ident(name))

def _coerced_lit(value: Any, target_dtype: str):
    """Wrap a user-provided value as a literal cast to the target column's type.

    UI always feeds text per the operator spec; this is where the
    type coercion happens so the per-action UI stays simple.
    """
    return F.lit(value).cast(target_dtype)

def _apply_formula(df, action: Dict[str, Any]):
    target = action.get("target", "")
    expression = action.get("expression", "")
    if not target:
        raise ValueError("formula: 'target' is required")
    if not expression:
        raise ValueError("formula: 'expression' is required")
    return df.withColumn(target, F.expr(expression))

def _apply_cast(df, action: Dict[str, Any]):
    column = action.get("column", "")
    to_type = action.get("to", "")
    on_error = action.get("on_error", "null")
    _require_column(df, column, "cast")
    if not to_type:
        raise ValueError("cast: 'to' (target type) is required")
    if on_error == "null":
        return df.withColumn(
            column,
            F.expr(f"try_cast({_quoted_ident(column)} as {to_type})"),
        )
    return df.withColumn(column, _col(column).cast(to_type))

def _apply_replace_value(df, action: Dict[str, Any]):
    column = action.get("column", "")
    match_mode = action.get("match_mode", "exact")
    match = action.get("match", "")
    with_val = action.get("with", "")
    case_sensitive = bool(action.get("case_sensitive", False))
    _require_column(df, column, "replace_value")
    if match_mode not in VALID_MATCH_MODES:
        raise ValueError(
            f"replace_value: unsupported match_mode '{match_mode}'. "
            f"Choose one of: {sorted(VALID_MATCH_MODES)}"
        )
    dtype = _column_dtype(df, column)
    col_as_string = _col(column).cast("string")
    match_lit = F.lit(match)
    if case_sensitive:
        cmp_col = col_as_string
        cmp_lit = match_lit
    else:
        cmp_col = F.lower(col_as_string)
        cmp_lit = F.lower(match_lit)

    if match_mode == "exact":
        predicate = cmp_col == cmp_lit
    elif match_mode == "contains":
        predicate = cmp_col.contains(cmp_lit)
    elif match_mode == "prefix":
        predicate = cmp_col.startswith(cmp_lit)
    elif match_mode == "suffix":
        predicate = cmp_col.endswith(cmp_lit)
    else:  # regex
        pattern = match if case_sensitive else f"(?i){match}"
        predicate = col_as_string.rlike(pattern)

    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(predicate, replacement).otherwise(_col(column)),
    )

def _apply_fill_null(df, action: Dict[str, Any]):
    column = action.get("column", "")
    with_val = action.get("with", "")
    _require_column(df, column, "fill_null")
    dtype = _column_dtype(df, column)
    replacement = _coerced_lit(with_val, dtype)
    return df.withColumn(
        column,
        F.when(_col(column).isNull(), replacement).otherwise(_col(column)),
    )

def _apply_text_case(df, action: Dict[str, Any]):
    column = action.get("column", "")
    case = action.get("case", "")
    _require_column(df, column, "text_case")
    fn = TEXT_CASE_FUNCTIONS.get(case)
    if fn is None:
        raise ValueError(
            f"text_case: unsupported case '{case}'. "
            f"Choose one of: {sorted(TEXT_CASE_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_trim(df, action: Dict[str, Any]):
    column = action.get("column", "")
    side = action.get("side", "both")
    _require_column(df, column, "trim")
    fn = TRIM_FUNCTIONS.get(side)
    if fn is None:
        raise ValueError(
            f"trim: unsupported side '{side}'. "
            f"Choose one of: {sorted(TRIM_FUNCTIONS.keys())}"
        )
    return df.withColumn(column, fn(_col(column)))

def _apply_regex_replace(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    replacement = action.get("replacement", "")
    _require_column(df, column, "regex_replace")
    return df.withColumn(
        column,
        F.regexp_replace(_col(column), pattern, replacement),
    )

def _apply_extract(df, action: Dict[str, Any]):
    column = action.get("column", "")
    pattern = action.get("pattern", "")
    target = action.get("target") or column
    group_raw = action.get("group", 0)
    _require_column(df, column, "extract")
    try:
        group_idx = int(group_raw)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"extract: 'group' must be an integer, got {group_raw!r}"
        ) from exc
    return df.withColumn(
        target,
        F.regexp_extract(_col(column), pattern, group_idx),
    )

def _apply_parse_date(df, action: Dict[str, Any]):
    column = action.get("column", "")
    kind = action.get("kind", "date")
    fmt = action.get("format") or None
    on_error = action.get("on_error", "null")
    _require_column(df, column, "parse_date")
    if kind not in ("date", "timestamp"):
        raise ValueError(
            f"parse_date: unsupported kind '{kind}'. Choose date or timestamp."
        )
    # PySpark exposes F.try_to_timestamp from Spark 3.5 but only adds
    # F.try_to_date in Spark 4.0. Current Databricks Runtimes still ship
    # Spark 3.5, where referencing F.try_to_date raises AttributeError
    # before any data is touched. Route the null-on-error path through
    # primitives that exist in both 3.5 and 4.0:
    #   - F.try_to_timestamp  (PySpark 3.5+, returns NULL under ANSI too)
    #   - SQL try_cast        (Spark 3.5+, returns NULL under ANSI too)
    #   - F.to_date           (PySpark 3.5+; returns NULL on parse failure
    #                          under the default non-ANSI mode, which is
    #                          the Databricks default. Under ANSI mode it
    #                          raises — accepted limitation until
    #                          try_to_date lands on every supported DBR.)
    if on_error == "null":
        if kind == "timestamp":
            if fmt:
                return df.withColumn(
                    column, F.try_to_timestamp(_col(column), F.lit(fmt))
                )
            return df.withColumn(column, F.try_to_timestamp(_col(column)))
        # kind == "date"
        if fmt:
            return df.withColumn(column, F.to_date(_col(column), fmt))
        return df.withColumn(
            column, F.expr(f"try_cast({_quoted_ident(column)} as date)")
        )
    # on_error == "error": surface parse failures as Spark exceptions.
    fn = F.to_date if kind == "date" else F.to_timestamp
    if fmt:
        return df.withColumn(column, fn(_col(column), fmt))
    return df.withColumn(column, fn(_col(column)))

ACTION_DISPATCH: Dict[str, Callable] = {
    "formula": _apply_formula,
    "cast": _apply_cast,
    "replace_value": _apply_replace_value,
    "fill_null": _apply_fill_null,
    "text_case": _apply_text_case,
    "trim": _apply_trim,
    "regex_replace": _apply_regex_replace,
    "extract": _apply_extract,
    "parse_date": _apply_parse_date,
}

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    actions: List[Dict[str, Any]] = config.get("actions", []) or []

    if not actions:
        return {"prepared_data": df}

    for index, action in enumerate(actions):
        if not isinstance(action, dict):
            raise ValueError(
                f"actions[{index}]: expected an object, got {type(action).__name__}"
            )
        if action.get("enabled", True) is False:
            continue
        action_type = action.get("type", "")
        fn = ACTION_DISPATCH.get(action_type)
        if fn is None:
            raise ValueError(
                f"actions[{index}]: unsupported action type {action_type!r}. "
                f"Choose one of: {sorted(ACTION_DISPATCH.keys())}"
            )
        df = fn(df, action)
    return {"prepared_data": df}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "actions": [
        {
            "type": "formula",
            "target": "month_year_parts",
            "expression": "SPLIT(month_year, '-')"
        },
        {
            "type": "formula",
            "target": "sort_year",
            "expression": "CAST(element_at(month_year_parts, 2) AS INT)"
        },
        {
            "type": "formula",
            "target": "sort_month",
            "expression": "CAST(element_at(month_year_parts, 1) AS INT)"
        }
    ]
}
inputs = {
    "data": ctx["aggregate_monthly_order_status.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["prepare_month_year_parts.prepared_data"] = out["prepared_data"]

In [0]:
"""
id: sort_monthly_order_status
template: sort
templateVersion: 1.0.0
name: SortedMonthlyOrderStatus
position:
  x: 2707.9387994311505
  y: 274.2219127149878
description:
  text: Sort data by month of the order date in descending order.
  hash: 6793b707
previewCodeHash: 068c82f9c5915f7c
previewMode: "1000"
config:
  sort_expressions:
    - columnExpr:
        expr: OrderCount
        type: column
      sortBy: DESC
    - columnExpr:
        expr: month_year
        type: column
      sortBy: DESC
    - columnExpr:
        expr: order_status
        type: column
      sortBy: DESC
input:
  - node: aggregate_monthly_order_status
    input_port: data
    output_port: aggregated_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "OrderCount",
                "type": "column"
            },
            "sortBy": "DESC"
        },
        {
            "columnExpr": {
                "expr": "month_year",
                "type": "column"
            },
            "sortBy": "DESC"
        },
        {
            "columnExpr": {
                "expr": "order_status",
                "type": "column"
            },
            "sortBy": "DESC"
        }
    ]
}
inputs = {
    "data": ctx["aggregate_monthly_order_status.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["sort_monthly_order_status.sorted_data"] = out["sorted_data"]

In [0]:
"""
id: sort_by_year_month
template: sort
templateVersion: 1.0.0
name: SortByYearMonth
position:
  x: 2999.3498849829066
  y: 236.91001575486038
description:
  text: Sort data descending by year, then ascending by month.
  hash: 7787f256
previewCodeHash: 32b490d617fee72f
previewMode: "1000"
config:
  sort_expressions:
    - columnExpr:
        expr: sort_year
        type: column
      sortBy: DESC
    - columnExpr:
        expr: sort_month
        type: column
      sortBy: ASC
input:
  - node: prepare_month_year_parts
    input_port: data
    output_port: prepared_data
"""

# generated from the system
from typing import Dict, Any
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    sort_expressions = config.get("sort_expressions", [])

    if not sort_expressions:
        return {"sorted_data": df}

    order_cols = []
    for sort_def in sort_expressions:
        col_expr = sort_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        direction = sort_def.get("sortBy", "UNSET")

        col = F.col(raw_expr)
        if direction == "DESC":
            col = col.desc()
        elif direction == "ASC":
            col = col.asc()

        order_cols.append(col)

    return {"sorted_data": df.orderBy(*order_cols)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "sort_expressions": [
        {
            "columnExpr": {
                "expr": "sort_year",
                "type": "column"
            },
            "sortBy": "DESC"
        },
        {
            "columnExpr": {
                "expr": "sort_month",
                "type": "column"
            },
            "sortBy": "ASC"
        }
    ]
}
inputs = {
    "data": ctx["prepare_month_year_parts.prepared_data"]
}
out = run(config, inputs, spark)
ctx["sort_by_year_month.sorted_data"] = out["sorted_data"]

In [0]:
"""
id: output_21
template: output
templateVersion: 1.0.0
name: lakeflow_designer.enr.OrderCountbyMonth
position:
  x: 3005.2031589809567
  y: 388.6757017049183
description:
  text: Save the data as a new table named output_21, replacing existing data if any.
  hash: add6f3c4
previewMode: "1000"
config:
  catalog: lakeflow_designer
  schema: enr
  table_name: OrderCountbyMonth
input:
  - node: sort_monthly_order_status
    input_port: data
    output_port: sorted_data
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "lakeflow_designer",
    "schema": "enr",
    "table_name": "OrderCountbyMonth"
}
inputs = {
    "data": ctx["sort_monthly_order_status.sorted_data"]
}
out = run(config, inputs, spark)

In [0]:
"""
id: select_month_year_order_status
template: transform
templateVersion: 2.0.0
name: SelectMonthYearOrderStatus
position:
  x: 3270.123531587503
  y: 124.63727745433197
description:
  text: Keep only month_year, order_status, and OrderCount columns in the specified order.
  hash: 3610e7be
previewCodeHash: 1b70d98fc74a4fb9
previewMode: "1000"
config:
  mode: select
  edits:
    - column: month_year
    - column: order_status
    - column: OrderCount
  ordered:
    - month_year
    - order_status
    - OrderCount
input:
  - node: sort_by_year_month
    input_port: data
    output_port: sorted_data
"""

# generated from the system
from typing import Any, Dict, List
from pyspark.sql import functions as F

def _col_ref(name: str):
    if "." in name and not (name.startswith("`") and name.endswith("`")):
        return F.col("`" + name.replace("`", "``") + "`")
    return F.col(name)

def _is_checked(item: Dict[str, Any]) -> bool:
    return item.get("checked", True) is not False

def _column_for(item: Dict[str, Any]):
    col = _col_ref(item.get("column", ""))
    alias = item.get("alias")
    if alias:
        col = col.alias(alias)
    return col

def _passthrough_column(name: str, rename_map: Dict[str, Dict[str, Any]]):
    entry = rename_map.get(name)
    col = _col_ref(name)
    if entry and entry.get("alias"):
        col = col.alias(entry["alias"])
    return col

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    mode = config.get("mode", "passthrough")
    edits: List[Dict[str, Any]] = config.get("edits", [])
    ordered: List[str] = config.get("ordered") or []

    if mode == "select":
        checked_edits = [item for item in edits if _is_checked(item)]
        if not checked_edits:
            return {"transformed_data": df}
        order_index = {name: i for i, name in enumerate(ordered)}
        tail = len(order_index)
        ordered_edits = sorted(
            checked_edits,
            key=lambda item: order_index.get(item.get("column", ""), tail),
        )
        return {"transformed_data": df.select(*(_column_for(item) for item in ordered_edits))}

    unchecked_cols = {
        item.get("column", "")
        for item in edits
        if not _is_checked(item)
    }
    rename_map = {
        item.get("column", ""): item
        for item in edits
        if item.get("alias") and _is_checked(item)
    }
    upstream_cols = list(df.columns)
    upstream_set = set(upstream_cols)

    effective_ordered = ordered if ordered else list(upstream_cols)

    placed = set()
    out = []
    for token in effective_ordered:
        if token in placed or token not in upstream_set:
            continue
        placed.add(token)
        if token in unchecked_cols:
            continue
        out.append(_passthrough_column(token, rename_map))

    for col in upstream_cols:
        if col in placed or col in unchecked_cols:
            continue
        out.append(_passthrough_column(col, rename_map))

    if not out:
        return {"transformed_data": df}
    return {"transformed_data": df.select(*out)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "mode": "select",
    "edits": [
        {
            "column": "month_year"
        },
        {
            "column": "order_status"
        },
        {
            "column": "OrderCount"
        }
    ],
    "ordered": [
        "month_year",
        "order_status",
        "OrderCount"
    ]
}
inputs = {
    "data": ctx["sort_by_year_month.sorted_data"]
}
out = run(config, inputs, spark)
ctx["select_month_year_order_status.transformed_data"] = out["transformed_data"]